# Ipotesi 1 — VideoLLaMA 2 7B: il modello usa davvero il video?

Sulle 50 domande di `video_nutrition_estimation` si confrontano le risposte del modello
in tre condizioni. Domanda, opzioni e prompt sono sempre identici; cambia solo l'input visivo.

| Condizione | Input visivo | In questo notebook |
|---|---|---|
| `C0_video` | il video vero (8 frame) | **riusato** dal baseline, si rifanno solo 5 domande di controllo |
| `C1_testo` | nessuno: solo domanda e opzioni | run completo |
| `C3_scambio` | il video di un'altra ricetta, in cui nessuna opzione compare | run completo |

**Scaletta:** 1. ambiente · 2. percorsi e moduli · 3. controllo dei dati · 4. funzioni di
inferenza · 5. prova rapida · 6. run completi · 7. primo sguardo ai numeri

**Prima di iniziare** (vedi `README.md` in questa cartella): `scripts/sync_drive.sh` deve aver copiato
`src/` e `data/` su Drive, con il vecchio risultato di C0 copiato in `results/ipotesi_video/` come
`{modello}_C0_video_video_nutrition_estimation.jsonl`.

## 1. Ambiente (Colab L4)

**Runtime > Cambia tipo di runtime > GPU L4**, Python 3.12, sessione nuova.

Identico ai notebook del baseline: stesse versioni, stessa patch, stesso checkpoint.
Cambia solo l'installazione: repo fissato a un commit, `torchaudio` allineato a torch 2.7.0,
`peft` non installato (non serve), controlli espliciti su versioni e GPU.
Su L4 (24 GB) il modello occupa circa 15 GiB.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM

!nvidia-smi
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")
# Serve Python 3.12, come il baseline: su 3.13 tokenizers 0.19.1 non ha la wheel e la
# compilazione con Rust fallisce, quindi pip non installa transformers 4.40 ne' decord.
assert sys.version_info[:2] == (3, 12), (
    f"Python {sys.version.split()[0]}: serve 3.12 (come il baseline). Runtime > Cambia tipo di runtime > "
    "Versione runtime: sceglierne una con Python 3.12, poi riavviare la sessione.")

In [ ]:
# PyTorch 2.7.0, come il baseline. torchaudio va allineato: quello preinstallato da Colab
# e' compilato per un altro torch e non si potrebbe importare.
%pip install -q torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Repo ufficiale di VideoLLaMA 2, copia pulita fissata a un commit: se il repo cambia,
# il notebook usa comunque lo stesso codice del baseline.
import os, shutil, sys
VL2_DIR = "/content/VideoLLaMA2"
VL2_COMMIT = "c0bb03abf6b8a6b9a8dccac006fb4db5d4d9e414"   # ultimo commit del repo (gennaio 2025)

%cd /content
if os.path.exists(VL2_DIR):
    shutil.rmtree(VL2_DIR)          # toglie anche le patch di una sessione precedente
!git clone -q https://github.com/DAMO-NLP-SG/VideoLLaMA2.git {VL2_DIR}
!git -C {VL2_DIR} checkout -q {VL2_COMMIT}
!git -C {VL2_DIR} log -1 --format="VideoLLaMA2: commit %h del %cd"

if VL2_DIR not in sys.path:
    sys.path.insert(0, VL2_DIR)
%cd /content/VideoLLaMA2

In [ ]:
%pip install -q --upgrade pip setuptools wheel   # servono a compilare tokenizers se Python e' 3.13

In [ ]:
# Versioni ufficiali di VideoLLaMA2 (pyproject.toml del repo):
#   transformers==4.40.0, tokenizers==0.19.1, accelerate==0.26.1, decord==0.6.0
# - tokenizers 0.19.1 ha la wheel per Python 3.12 ma non per 3.13: li' si compila con Rust (alcuni minuti).
# - huggingface_hub < 1.0 e' richiesto da transformers 4.40 (Colab ha la 1.x).
# - timm < 1.1: il repo importa timm.models.layers, deprecato.
# - peft NON si installa: VideoLLaMA2 non lo usa in inferenza, e una versione nuova
#   potrebbe trascinare un aggiornamento di transformers.
# Gli avvisi rossi di pip su gradio, sentence-transformers, peft o torchao sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
#
# %pip non si ferma se l'installazione fallisce: se un pacchetto (es. tokenizers) non si
# compila, pip annulla TUTTO il comando e restano le versioni preinstallate di Colab.
# Per questo si usa pip() qui sotto, che si ferma e mostra l'errore vero.
import sys, os, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-60:]))
        raise RuntimeError(f"pip install fallito: {' '.join(args)} (errore qui sopra)")

print("Python:", sys.version)

if sys.version_info >= (3, 13):
    print("Python 3.13: compilo tokenizers 0.19.1 con Rust (alcuni minuti)...")
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
    os.environ["PATH"] += ":/root/.cargo/bin"
    os.environ["PYO3_USE_ABI3_FORWARD_COMPATIBILITY"] = "1"   # pyo3 di tokenizers 0.19 non conosce ancora 3.13
    pip("tokenizers==0.19.1", "--no-binary", "tokenizers")
else:
    pip("tokenizers==0.19.1")

pip("transformers==4.40.0", "tokenizers==0.19.1", "accelerate==0.26.1", "huggingface_hub>=0.23,<1.0",
    "timm>=1.0.3,<1.1", "einops", "decord==0.6.0", "av", "imageio", "opencv-python")

# Controllo: se le versioni non sono quelle del baseline ci si ferma qui
from importlib.metadata import version, PackageNotFoundError
def _v(k):
    try:
        return version(k)
    except PackageNotFoundError:
        return "NON INSTALLATO"
_att = {"torch": "2.7.0", "transformers": "4.40.0", "tokenizers": "0.19.1",
        "accelerate": "0.26.1", "decord": "0.6.0"}
_vers = {k: _v(k) for k in _att}
print(_vers)
_err = {k: v for k, v in _vers.items() if not v.startswith(_att[k])}
assert not _err, f"versioni errate {_err}, attese {_att}: controllare l'output di pip sopra"

In [ ]:
import os
# Evita il lookup del secret HF_TOKEN di Colab che va in timeout e "appende" il download
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # download piu' stabile
os.environ.pop("HF_TOKEN", None)

import gc, sys, pathlib, torch
# Se torch era gia' stato importato prima dell'installazione, in memoria c'e' ancora quello vecchio
assert torch.__version__.startswith("2.7.0"), (
    f"torch {torch.__version__} in memoria, atteso 2.7.0: Runtime > Riavvia sessione, poi rieseguire dall'inizio")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

# Libero la VRAM da tentativi precedenti
for name in ['model', 'processor', 'tokenizer']:
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM libera: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

# -----------------------------------------------------------------------------
# PATCH encoder.py: il repo forza a mano config._attn_implementation="flash_attention_2"
# sulla CLIP/Siglip vision tower, ma questi modelli non supportano flash-attn -> ValueError.
# Sostituiamo con "eager" DIRETTAMENTE nel file sorgente, prima di importare videollama2.
# -----------------------------------------------------------------------------
_enc = pathlib.Path("/content/VideoLLaMA2/videollama2/model/encoder.py")
_txt = _enc.read_text()
if '"flash_attention_2"' in _txt:
    _enc.write_text(_txt.replace('"flash_attention_2"', '"eager"'))
    print("Patch applicata: flash_attention_2 -> eager in encoder.py")
else:
    print("Nessuna patch necessaria (flash_attention_2 non trovato in encoder.py)")

# Svuoto la cache dei moduli gia' importati, cosi' l'import qui sotto rilegge il file patchato
for _m in list(sys.modules):
    if _m.startswith("videollama2"):
        del sys.modules[_m]

# 1) Scarico lo snapshot VideoLLaMA2-7B (circa 16 GB)
from huggingface_hub import snapshot_download
model_path = snapshot_download("DAMO-NLP-SG/VideoLLaMA2-7B")
print("Modello scaricato in:", model_path)

# 2) Inizializzo passando il path LOCALE (niente altre chiamate di rete).
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
print("Inizio caricamento modello...")
model, processor, tokenizer = videollama2_model_init(
    model_path,
    device_map={"": device},
    attn_implementation="eager",   # forza eager sul modello linguistico
    use_flash_attn=False,
)

# imposto il modello in modalita' inferenza
model.eval()
print("Modello VideoLLaMA2-7B caricato con successo - dtype:", next(model.parameters()).dtype)
print("num_frames usati dal processor:", getattr(model.config, "num_frames", "default (8)"))
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

## 2. Percorsi e moduli

I moduli si importano da `src/hdepic/` su Drive: niente `%%writefile`.
`hdepic.eval` e' la stessa versione usata per il baseline.

In [ ]:
import os, sys, glob, json, importlib

BASE    = "/content/drive/MyDrive/Progetto Deep Learning 1"   # cartella del progetto su Drive
SRC      = os.path.join(BASE, "src")                            # codice, copiato su Drive da scripts/sync_drive.sh
DERIVATI = os.path.join(BASE, "data", "derived", "ipotesi_video")
ANN_ORIG = os.path.join(BASE, "data", "hd-epic")                # annotazioni originali del benchmark
ANN_SWAP = os.path.join(DERIVATI, "annotazioni_swap")           # annotazioni con il video scambiato (C3)
OUT_DIR  = os.path.join(BASE, "results", "ipotesi_video")
os.makedirs(OUT_DIR, exist_ok=True)
MODELLO  = "videollama2"

sys.path.insert(0, SRC)
from hdepic import eval as H
from hdepic import ablazione as A
importlib.reload(H); importlib.reload(A)
print(f"hdepic.eval {H.VERSION} | ablazione {A.VERSION}")

with open(os.path.join(DERIVATI, "swap_map.json")) as f:
    SWAP_MAP = json.load(f)

VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
print("cartelle video:", [os.path.basename(d) for d in VIDEO_ROOTS])
print(f"video campionati indicizzati: {len(VIDEO_INDEX['sampled'])}")

c0 = A.percorso(OUT_DIR, MODELLO, "C0_video")
print(f"\nC0 (video vero): {'trovato' if os.path.exists(c0) else 'MANCANTE'} -> {os.path.basename(c0)}")

## 3. Controllo dei dati

Prima di girare si verifica che:
- le annotazioni scambiate differiscano da quelle originali **solo** per l'id del video;
- tutti i video servano davvero: quelli originali (stesse domande di C0) e quelli di scambio.

Le attese sono **50 domande utilizzabili** in entrambe le tabelle. Se ne mancano, il confronto
con C0 resta valido ma va fatto sulle sole domande comuni (lo gestisce `03_analisi`).

In [ ]:
READABLE = {}   # cache: aprire i video da Drive e' lento, si controlla una volta sola

A.controlla_annotazioni_scambiate(ANN_ORIG, ANN_SWAP)
for nome, ann_dir in [("annotazioni originali (C1, controllo C0)", ANN_ORIG),
                      ("annotazioni scambiate (C3)", ANN_SWAP)]:
    cov, dettagli = H.preflight(ann_dir, VIDEO_INDEX, tasks=[A.TASK], cache=READABLE)
    print(f"\n{nome}")
    print(cov.to_string(index=False))
    rotti = sorted({v for ids in dettagli[A.TASK].values() for v in ids})
    if rotti:
        print("  file mancanti o non decodificabili:", rotti)

## 4. Funzioni di inferenza

- **`infer_video`** e' identica a `videollama2_infer` del baseline: serve per C3 e per il controllo di C0.
- **`infer_testo`** usa la modalita' `text` di `mm_infer`, prevista dal repo ufficiale: nessun tensore
  visivo e nessun token `<video>` nel prompt. Il resto (system prompt del modello, decoding greedy,
  `max_new_tokens`) e' identico.

In [ ]:
import numpy as np
from videollama2 import mm_infer

MAX_NEW_TOKENS = 32                                   # la risposta attesa e' una lettera
NUM_FRAMES = getattr(model.config, "num_frames", 8)

def sottocampiona(paths, n=NUM_FRAMES):
    """Liste di immagini -> n elementi uniformi (process_video ricampiona solo i path mp4)."""
    if len(paths) <= n:
        return paths
    sel = np.linspace(0, len(paths) - 1, n, dtype=int)
    return [paths[i] for i in sel]

def infer_video(visual, prompt):
    """visual: path di un mp4 oppure lista di path di immagini."""
    if isinstance(visual, list):
        visual = sottocampiona(visual)
    tensor = processor["video"](visual)
    assert tensor.shape[0] == NUM_FRAMES, f"attesi {NUM_FRAMES} elementi visivi, arrivati {tensor.shape[0]}"
    return mm_infer(tensor, prompt, model, tokenizer, modal="video",
                    do_sample=False, max_new_tokens=MAX_NEW_TOKENS)

def infer_testo(prompt):
    """Solo testo: nessun input visivo."""
    return mm_infer(None, prompt, model, tokenizer, modal="text",
                    do_sample=False, max_new_tokens=MAX_NEW_TOKENS)

# condizione -> (funzione di inferenza, annotazioni da usare)
INFER = {"C1_testo": A.solo_testo(infer_testo), "C3_scambio": infer_video}
ANN   = {"C1_testo": ANN_ORIG,                  "C3_scambio": ANN_SWAP}
print(f"pronto - {NUM_FRAMES} frame per domanda col video, decoding greedy")

## 5. Prova rapida

Tre domande per condizione, scritte in file `*_prova_*` separati dai run completi.

**Cosa controllare a occhio:**
- **C1** le risposte sono lettere (`A`–`E`) e `parse_method` e' `leading_letter`;
- **C3** il video indicato e' diverso dall'originale tra parentesi;
- nessuna riga con `ERRORE`.

In [ ]:
prova = {c: A.esegui(MODELLO, c, INFER[c], ANN[c], VIDEO_INDEX, OUT_DIR,
                     limit=3, prova=True, readable_cache=READABLE)
         for c in ["C1_testo", "C3_scambio"]}
A.mostra_prova(prova, SWAP_MAP)

### Controllo dell'ambiente su C0

C0 non viene rifatto: si riusano i risultati del baseline. Per essere sicuri che questa
sessione Colab si comporti come quella del baseline, si rifanno **5 domande col video vero**.
Con il decoding greedy le lettere devono essere **identiche** a quelle salvate in C0.

In [ ]:
ctrl = A.esegui(MODELLO, "C0_controllo", infer_video, ANN_ORIG, VIDEO_INDEX, OUT_DIR,
                limit=5, readable_cache=READABLE)
_ = A.confronta_controllo(A.percorso(OUT_DIR, MODELLO, "C0_video"),
                          A.percorso(OUT_DIR, MODELLO, "C0_controllo"))

## 6. Run completi

Ogni risposta viene scritta subito sul `.jsonl`: se Colab si disconnette, rilanciare la
cella riprende dalla domanda successiva.

In [ ]:
risultati = {c: A.esegui(MODELLO, c, INFER[c], ANN[c], VIDEO_INDEX, OUT_DIR,
                         readable_cache=READABLE)
             for c in ["C1_testo", "C3_scambio"]}

## 7. Primo sguardo ai numeri

Solo per questo modello. L'analisi completa, con entrambi i modelli, i grafici e gli esempi,
e' in `03_analisi.ipynb`.

In [ ]:
df = A.controlla_allineamento(A.carica_risultati(OUT_DIR, [MODELLO]))
tab = A.tabella_principale(df)
display(tab)
_ = A.verdetto(tab)